# VCF deepfake training (cached frames, 2x T4)
**Before running:** Accelerator = GPU T4 x2. Add inputs: `fml-deepfake-project` and `preprocessed-videos-cache`.

In [8]:
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

cv2.setNumThreads(1)

LABELS = {
    "targets": 0,
    "simswap_224": 1,
    "simswap_512": 1,
    "deeplivecam": 1,
    "deeplivecam_enhance": 1,
}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
MEAN = np.array([0.43216, 0.394666, 0.37645], dtype=np.float32)
STD = np.array([0.22803, 0.22145, 0.216989], dtype=np.float32)


def get_label(relative_path: Path) -> int:
    for part in relative_path.parts:
        if part in LABELS:
            return LABELS[part]
    raise ValueError(f"Cannot determine real/fake label for: {relative_path}")


def source_id(relative_path: Path) -> str:
    # Assumes the prefix before the first underscore identifies the source.
    return relative_path.stem.split("_")[0]


def read_video(path: Path, num_frames: int = 8, image_size: int = 112) -> torch.Tensor:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        raise RuntimeError(f"Video has no readable frames: {path}")

    indices = np.linspace(0, frame_count - 1, num=min(num_frames, frame_count)).astype(int)
    frames = []

    for index in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        success, frame = cap.read()
        if not success:
            cap.release()
            raise RuntimeError(f"Could not read frame {index} from {path}")

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        height, width = frame.shape[:2]
        scale = image_size / min(height, width)
        new_width = max(image_size, round(width * scale))
        new_height = max(image_size, round(height * scale))
        frame = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_AREA)

        top = (new_height - image_size) // 2
        left = (new_width - image_size) // 2
        frame = frame[top:top + image_size, left:left + image_size]

        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - MEAN) / STD)

    cap.release()
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())

    video = np.transpose(np.stack(frames, axis=0), (3, 0, 1, 2)).copy()
    return torch.from_numpy(video).float()


class VCFDataset(Dataset):
    """
    Each item contains raw/c23/c40 versions of one balanced-dataset video.
    Only records with all three matching relative paths are retained.
    """
    def __init__(self, root="vcf-balanced", num_frames=8, image_size=112, records=None):
        self.root = Path(root)
        self.num_frames = num_frames
        self.image_size = image_size

        if not self.root.exists():
            raise FileNotFoundError(f"Dataset directory not found: {self.root.resolve()}")

        self.records = records if records is not None else self._discover_records()
        if not self.records:
            raise RuntimeError(
                f"No videos have matching raw/c23/c40 paths under {self.root}. "
                "See the per-split filename comparison."
            )

    @staticmethod
    def _video_map(folder: Path):
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing split directory: {folder}")
        return {
            p.relative_to(folder).as_posix(): p
            for p in folder.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        }

    def _discover_records(self):
        maps = {
            split: self._video_map(self.root / split)
            for split in ("raw", "c23", "c40")
        }

        names_raw = set(maps["raw"])
        names_c23 = set(maps["c23"])
        names_c40 = set(maps["c40"])
        common = names_raw & names_c23 & names_c40

        print("Video file counts:")
        for split, video_map in maps.items():
            print(f"  {split}: {len(video_map)}")
        print(f"  common relative paths: {len(common)}")
        print(f"  raw missing c23: {len(names_raw - names_c23)}")
        print(f"  raw missing c40: {len(names_raw - names_c40)}")
        print(f"  c23 files without raw: {len(names_c23 - names_raw)}")
        print(f"  c40 files without raw: {len(names_c40 - names_raw)}")

        records = []
        for name in sorted(common):
            relative_path = Path(name)
            paths = {split: maps[split][name] for split in ("raw", "c23", "c40")}
            records.append({
                "paths": paths,
                "label": get_label(relative_path),
                "group_id": source_id(relative_path),
                "relative_path": name,
            })

        n_real = sum(record["label"] == 0 for record in records)
        n_fake = sum(record["label"] == 1 for record in records)
        print(f"Matched records retained: {len(records)} | real={n_real} | fake={n_fake}")

        if len(common) < len(names_raw):
            print(
                "WARNING: some raw records were skipped because the same relative "
                "path was absent from c23 or c40."
            )

        return records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        raw_video = read_video(record["paths"]["raw"], self.num_frames, self.image_size)
        c23_video = read_video(record["paths"]["c23"], self.num_frames, self.image_size)
        c40_video = read_video(record["paths"]["c40"], self.num_frames, self.image_size)
        label = torch.tensor(record["label"], dtype=torch.float32)
        return raw_video, c23_video, c40_video, label, record["group_id"]

In [15]:
import torch.nn.functional as F


def vcf_loss(
    logits_raw,
    logits_c23,
    logits_c40,
    features_raw,
    features_c23,
    features_c40,
    labels,
    lambda_pred=0.4,
):
    """
    logits_*: (B, 1)
    features_*: (B, 512)
    labels: (B, 1), real=0 and fake=1
    Standard BCE (no manual class weight) + cosine consistency across compressions.
    """
    labels = labels.float()

    loss_raw = F.binary_cross_entropy_with_logits(logits_raw.float(), labels)
    loss_c23 = F.binary_cross_entropy_with_logits(logits_c23.float(), labels)
    loss_c40 = F.binary_cross_entropy_with_logits(logits_c40.float(), labels)
    loss_bce = (loss_raw + loss_c23 + loss_c40) / 3.0

     # Compute probabilities in float32 for numerical stability.
    prob_raw = torch.sigmoid(logits_raw.float())
    prob_c23 = torch.sigmoid(logits_c23.float())
    prob_c40 = torch.sigmoid(logits_c40.float())

    
    pred_raw_c23 = F.mse_loss(prob_raw, prob_c23)
    pred_raw_c40 = F.mse_loss(prob_raw, prob_c40)
    pred_c23_c40 = F.mse_loss(prob_c23, prob_c40)
    loss_pred = 2*(pred_raw_c23 + pred_raw_c40 + pred_c23_c40) / 3.0

    total_loss = loss_bce + lambda_pred * loss_pred
    return total_loss, {
        "bce": loss_bce.detach(),
        "pred_consistency": loss_pred.detach(),
    }

In [16]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights


class VCFDeepfakeDetector(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        self.backbone = r3d_18(weights=weights)
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Linear(512, 1)

    def forward(self, video):
        """
        video: (B, C, T, H, W), RGB. Returns logits (B, 1) and features (B, 512).
        autocast is applied inside forward because autocast state is thread-local
        and DataParallel runs each GPU replica in its own thread.
        """
        with torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=video.is_cuda
        ):
            features = self.backbone(video)
            logits = self.classifier(features)
        return logits, features

In [17]:
import os

CACHE = None
for root, dirs, files in os.walk("/kaggle/input"):
    if {"raw", "c23", "c40"}.issubset(dirs):
        n = sum(1 for _ in Path(root).rglob("*.npy"))
        print(root, "-> npy files:", n)
        if n > 0 and CACHE is None:
            CACHE = Path(root)

assert CACHE is not None, "Cache not found. Add the 'preprocessed-videos-cache' dataset as an input."
print("\nUsing CACHE =", CACHE)

/kaggle/input/datasets/thirteenthdev/preprocessed-videos-cache -> npy files: 4800
/kaggle/input/datasets/abhinavreddy16/fml-deepfake-project/vcf-balanced/vcf-balanced -> npy files: 0

Using CACHE = /kaggle/input/datasets/thirteenthdev/preprocessed-videos-cache


In [18]:
def cache_path(split, rel):
    return CACHE / split / (rel + ".npy")


def load_cached(path):
    x = np.load(path).astype(np.float32) / 255.0   # uint8 -> 0..1
    x = (x - MEAN) / STD                           # same normalization as read_video
    return torch.from_numpy(np.transpose(x, (3, 0, 1, 2)).copy())


def _getitem(self, index):
    r = self.records[index]
    vids = [load_cached(cache_path(s, r["relative_path"])) for s in ("raw", "c23", "c40")]
    label = torch.tensor(r["label"], dtype=torch.float32)
    return vids[0], vids[1], vids[2], label, r["group_id"]


VCFDataset.__getitem__ = _getitem

In [19]:
DATA_ROOT = Path("/kaggle/input/datasets/abhinavreddy16/fml-deepfake-project/vcf-balanced/vcf-balanced")

_ds = VCFDataset(root=DATA_ROOT, num_frames=8, image_size=112)
_r = _ds.records[0]
_a = read_video(_r["paths"]["raw"], 8, 112)                    # original path
_b = load_cached(cache_path("raw", _r["relative_path"]))       # cached path
print(_a.shape, _b.shape, "match:", torch.allclose(_a, _b, atol=1e-5))
print("mean/std of cached sample:", _b.mean().item(), _b.std().item())
del _ds, _r, _a, _b

Video file counts:
  raw: 1600
  c23: 1600
  c40: 1600
  common relative paths: 1600
  raw missing c23: 0
  raw missing c40: 0
  c23 files without raw: 0
  c40 files without raw: 0
Matched records retained: 1600 | real=640 | fake=960
torch.Size([3, 8, 112, 112]) torch.Size([3, 8, 112, 112]) match: True
mean/std of cached sample: 0.7701799869537354 0.9296184778213501


In [22]:
import csv
import json
import random
import time

from torch.utils.data import DataLoader, Subset
from tqdm import tqdm


SEED = 42
NUM_FRAMES = 8
IMAGE_SIZE = 112
BATCH_SIZE = 4        # split across 2 GPUs -> 8 per GPU (use 8 if out of memory)
EPOCHS = 3
LEARNING_RATE = 1e-4
LAMBDA_PRED = 0.7
NUM_WORKERS = 4

ROOT = Path("/kaggle/working")
DATA_ROOT = Path("/kaggle/input/datasets/abhinavreddy16/fml-deepfake-project/vcf-balanced/vcf-balanced")
SPLIT_DIR = Path("/kaggle/input/datasets/abhinavreddy16/fml-deepfake-project/splits")
CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_splits(dataset):
    """
    Load the fixed split assignments created by src/create_splits.py.
    Never regenerate splits during model training.
    """
    split_dir = SPLIT_DIR / "splits"

    index_by_path = {
        record["relative_path"]: i
        for i, record in enumerate(dataset.records)
    }

    def load_indices(split_name):
        csv_path = split_dir / f"{split_name}.csv"
        if not csv_path.is_file():
            raise FileNotFoundError(
                f"Missing {csv_path}. Run `python src/create_splits.py` once before training."
            )

        indices = []
        seen_paths = set()

        with csv_path.open("r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            required = {"relative_path", "label", "group_id"}
            if not reader.fieldnames or not required.issubset(reader.fieldnames):
                raise ValueError(
                    f"{csv_path} must contain columns: relative_path,label,group_id"
                )

            for row in reader:
                relative_path = row["relative_path"]

                if relative_path in seen_paths:
                    raise ValueError(f"Duplicate path {relative_path!r} in {csv_path}")
                seen_paths.add(relative_path)

                if relative_path not in index_by_path:
                    raise ValueError(
                        f"Split record {relative_path!r} from {csv_path} "
                        "is missing from the current dataset."
                    )

                idx = index_by_path[relative_path]
                record = dataset.records[idx]

                if int(row["label"]) != int(record["label"]):
                    raise ValueError(f"Label mismatch for {relative_path!r} in {csv_path}")
                if row["group_id"] != str(record["group_id"]):
                    raise ValueError(
                        f"Group mismatch for {relative_path!r} in {csv_path}. "
                        "Check source_id() and recreate splits if needed."
                    )

                indices.append(idx)

        if not indices:
            raise ValueError(f"No records found in {csv_path}")

        return indices

    train_idx = load_indices("train")
    val_idx = load_indices("val")
    test_idx = load_indices("test")

    assert set(train_idx).isdisjoint(val_idx)
    assert set(train_idx).isdisjoint(test_idx)
    assert set(val_idx).isdisjoint(test_idx)

    groups = [record["group_id"] for record in dataset.records]
    train_groups = {groups[i] for i in train_idx}
    val_groups = {groups[i] for i in val_idx}
    test_groups = {groups[i] for i in test_idx}

    if not train_groups.isdisjoint(val_groups):
        raise ValueError("Source-group leakage between train and validation.")
    if not train_groups.isdisjoint(test_groups):
        raise ValueError("Source-group leakage between train and test.")
    if not val_groups.isdisjoint(test_groups):
        raise ValueError("Source-group leakage between validation and test.")

    labels = np.array([record["label"] for record in dataset.records])
    total = len(dataset)

    for name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
        n_real = int((labels[idx] == 0).sum())
        n_fake = int((labels[idx] == 1).sum())
        print(
            f"{name}: {len(idx)} samples ({100 * len(idx) / total:.2f}%) "
            f"| real={n_real} | fake={n_fake} "
            f"| groups={len(set(groups[i] for i in idx))}"
        )

    return train_idx, val_idx, test_idx


def run_epoch(model, loader, optimizer, device, scaler=None, phase="TRAIN"):
    training = optimizer is not None
    model.train(training)

    total_loss = total_bce = total_pred = 0.0
    correct = count = 0
    tp = tn = fp = fn = 0
    start_time = time.time()

    progress = tqdm(loader, desc=phase, unit="batch", dynamic_ncols=True)

    for batch_idx, (raw, c23, c40, labels, _group_ids) in enumerate(progress):
        raw = raw.to(device, non_blocking=True)
        c23 = c23.to(device, non_blocking=True)
        c40 = c40.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).view(-1, 1)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            # autocast happens inside the model's forward (DataParallel-safe)
            logits_raw, features_raw = model(raw)
            logits_c23, features_c23 = model(c23)
            logits_c40, features_c40 = model(c40)
            loss, parts = vcf_loss(
                logits_raw, logits_c23, logits_c40,
                features_raw, features_c23, features_c40,
                labels, lambda_pred=LAMBDA_PRED,
            )

            if training:
                if scaler is not None and device.type == "cuda":
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        avg_logits = (
            logits_raw.detach() + logits_c23.detach() + logits_c40.detach()
        ) / 3.0
        predictions = (avg_logits >= 0).float()
        correct += int((predictions == labels).sum().item())
        count += labels.numel()

        pred = predictions.view(-1).to(torch.int64)
        target = labels.view(-1).to(torch.int64)
        tp += int(((pred == 1) & (target == 1)).sum().item())
        tn += int(((pred == 0) & (target == 0)).sum().item())
        fp += int(((pred == 1) & (target == 0)).sum().item())
        fn += int(((pred == 0) & (target == 1)).sum().item())

        total_loss += float(loss.detach().item())
        total_bce += float(parts["bce"].item())
        total_pred += float(parts["pred_consistency"].item())

        progress.set_postfix(
            loss=f"{total_loss / (batch_idx + 1):.4f}",
            bce=f"{total_bce / (batch_idx + 1):.4f}",
            pred=f"{total_pred / (batch_idx + 1):.4f}",
            acc=f"{correct / max(count, 1):.3f}",
        )

    real_recall = tn / max(tn + fp, 1)
    fake_recall = tp / max(tp + fn, 1)
    balanced_acc = (real_recall + fake_recall) / 2.0
    print(
        f"\n[{phase}] TN={tn}, FP={fp}, FN={fn}, TP={tp} | "
        f"real recall={real_recall:.4f}, fake recall={fake_recall:.4f}, "
        f"balanced accuracy={balanced_acc:.4f} | "
        f"elapsed={(time.time() - start_time) / 60:.1f} min"
    )

    return {
        "loss": total_loss / max(len(loader), 1),
        "bce": total_bce / max(len(loader), 1),
        "pred_cons": total_pred / max(len(loader), 1),
        "accuracy": correct / max(count, 1),
        "real_recall": real_recall,
        "fake_recall": fake_recall,
        "balanced_accuracy": balanced_acc,
    }


def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    if device.type == "cuda":
        for i in range(torch.cuda.device_count()):
            print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print("Dataset root:", DATA_ROOT)
    print("Cache:", CACHE)

    dataset = VCFDataset(root=DATA_ROOT, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE)
    labels = np.array([record["label"] for record in dataset.records])
    print(
        f"Dataset records: {len(dataset)} | "
        f"real={(labels == 0).sum()} | fake={(labels == 1).sum()}"
    )

    train_idx, val_idx, test_idx = make_splits(dataset)
    print(f"Train: {len(train_idx)} | Validation: {len(val_idx)} | Test: {len(test_idx)}")

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=True,
        prefetch_factor=2,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        persistent_workers=True,
        prefetch_factor=2,
    )

    model = VCFDeepfakeDetector(pretrained=True).to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs (DataParallel)")
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_val_loss = float("inf")
    checkpoint_path = CHECKPOINT_DIR / "best_model_linear_fixed_split.pt"

    history = []
    RESULTS_DIR = ROOT / "results"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer, device, scaler, phase="TRAIN")
        with torch.no_grad():
            val_metrics = run_epoch(model, val_loader, None, device, phase="VALIDATION")

        print(
            f"\nEpoch {epoch}/{EPOCHS}\n"
            f"  Train: loss={train_metrics['loss']:.4f}, "
            f"BCE={train_metrics['bce']:.4f}, pred_cons={train_metrics['pred_cons']:.4f}, "
            f"acc={train_metrics['accuracy']:.4f}\n"
            f"  Val:   loss={val_metrics['loss']:.4f}, "
            f"BCE={val_metrics['bce']:.4f}, pred_cons={val_metrics['pred_cons']:.4f}, "
            f"acc={val_metrics['accuracy']:.4f}\n"
            f"  Val balanced accuracy={val_metrics['balanced_accuracy']:.4f}"
        )

        history.append({
            "epoch": epoch,
            **{f"train_{k}": float(v) for k, v in train_metrics.items()},
            **{f"val_{k}": float(v) for k, v in val_metrics.items()},
        })
        with open(RESULTS_DIR / "history.json", "w") as f:
            json.dump(history, f, indent=2)
        with open(RESULTS_DIR / "history.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=history[0].keys())
            writer.writeheader()
            writer.writerows(history)

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            inner = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(
                {
                    "model_state_dict": inner.state_dict(),   # no "module." prefix
                    "epoch": epoch,
                    "val_loss": best_val_loss,
                    "seed": SEED,
                    "num_frames": NUM_FRAMES,
                    "image_size": IMAGE_SIZE,
                    "lambda_cos": LAMBDA_COS,
                    "data_root": str(DATA_ROOT),
                },
                checkpoint_path,
            )
            print("  Saved best checkpoint:", checkpoint_path)

    print("\nTraining complete.")
    print("Best validation loss:", best_val_loss)
    print("Checkpoint:", checkpoint_path)
    print("The test set remains reserved for final evaluation.")


main()

Device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4
Dataset root: /kaggle/input/datasets/abhinavreddy16/fml-deepfake-project/vcf-balanced/vcf-balanced
Cache: /kaggle/input/datasets/thirteenthdev/preprocessed-videos-cache
Video file counts:
  raw: 1600
  c23: 1600
  c40: 1600
  common relative paths: 1600
  raw missing c23: 0
  raw missing c40: 0
  c23 files without raw: 0
  c40 files without raw: 0
Matched records retained: 1600 | real=640 | fake=960
Dataset records: 1600 | real=640 | fake=960
Train: 1050 samples (65.62%) | real=420 | fake=630 | groups=105
Validation: 150 samples (9.38%) | real=60 | fake=90 | groups=15
Test: 400 samples (25.00%) | real=160 | fake=240 | groups=40
Train: 1050 | Validation: 150 | Test: 400
Using 2 GPUs (DataParallel)


TRAIN: 100%|██████████| 263/263 [02:11<00:00,  2.00batch/s, acc=0.541, bce=0.7175, loss=0.7180, pred=0.0007]



[TRAIN] TN=101, FP=319, FN=163, TP=467 | real recall=0.2405, fake recall=0.7413, balanced accuracy=0.4909 | elapsed=2.2 min


VALIDATION: 100%|██████████| 38/38 [00:19<00:00,  1.99batch/s, acc=0.560, bce=0.7004, loss=0.7014, pred=0.0014]



[VALIDATION] TN=26, FP=34, FN=32, TP=58 | real recall=0.4333, fake recall=0.6444, balanced accuracy=0.5389 | elapsed=0.3 min

Epoch 1/3
  Train: loss=0.7180, BCE=0.7175, pred_cons=0.0007, acc=0.5410
  Val:   loss=0.7014, BCE=0.7004, pred_cons=0.0014, acc=0.5600
  Val balanced accuracy=0.5389
  Saved best checkpoint: /kaggle/working/checkpoints/best_model_linear_fixed_split.pt


...


[VALIDATION] TN=16, FP=44, FN=25, TP=65 | real recall=0.2667, fake recall=0.7222, balanced accuracy=0.4944 | elapsed=0.1 min

Epoch 2/3
  Train: loss=0.6712, BCE=0.6706, pred_cons=0.0008, acc=0.5867
  Val:   loss=0.7360, BCE=0.7334, pred_cons=0.0037, acc=0.5400
  Val balanced accuracy=0.4944


TRAIN: 100%|██████████| 263/263 [01:13<00:00,  3.56batch/s, acc=0.633, bce=0.6415, loss=0.6427, pred=0.0018]



[TRAIN] TN=132, FP=288, FN=97, TP=533 | real recall=0.3143, fake recall=0.8460, balanced accuracy=0.5802 | elapsed=1.2 min


VALIDATION: 100%|██████████| 38/38 [00:06<00:00,  5.82batch/s, acc=0.633, bce=1.1577, loss=1.1732, pred=0.0221]



[VALIDATION] TN=15, FP=45, FN=10, TP=80 | real recall=0.2500, fake recall=0.8889, balanced accuracy=0.5694 | elapsed=0.1 min

Epoch 3/3
  Train: loss=0.6427, BCE=0.6415, pred_cons=0.0018, acc=0.6333
  Val:   loss=1.1732, BCE=1.1577, pred_cons=0.0221, acc=0.6333
  Val balanced accuracy=0.5694

Training complete.
Best validation loss: 0.7013556321984843
Checkpoint: /kaggle/working/checkpoints/best_model_linear_fixed_split.pt
The test set remains reserved for final evaluation.


In [23]:
import csv
import json
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report,
)

CHECKPOINT_PATH = ROOT / "checkpoints" / "best_model_linear_fixed_split.pt"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def evaluate_test():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print(f"Cache: {CACHE}")

    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)

    dataset = VCFDataset(
        root=DATA_ROOT,
        num_frames=checkpoint["num_frames"],
        image_size=checkpoint["image_size"],
    )

    # Always use the persisted test split.
    split_file = SPLIT_DIR / "splits" / "test.csv"
    if not split_file.is_file():
        raise FileNotFoundError(f"Test split not found: {split_file}")

    index_by_path = {r["relative_path"]: i for i, r in enumerate(dataset.records)}
    test_indices, seen = [], set()

    with split_file.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        required = {"relative_path", "label", "group_id"}
        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(f"{split_file} must contain: relative_path,label,group_id")

        for row in reader:
            p = row["relative_path"]
            if p in seen:
                raise ValueError(f"Duplicate path {p!r} in {split_file}")
            seen.add(p)
            if p not in index_by_path:
                raise ValueError(f"Test record {p!r} missing from the current dataset.")
            idx = index_by_path[p]
            rec = dataset.records[idx]
            if int(row["label"]) != int(rec["label"]):
                raise ValueError(f"Label mismatch for {p!r}")
            if row["group_id"] != str(rec["group_id"]):
                raise ValueError(f"Group mismatch for {p!r}")
            test_indices.append(idx)

    if not test_indices:
        raise ValueError(f"No test records found in {split_file}")

    # shuffle=False, so output order matches test_indices
    test_loader = DataLoader(
        Subset(dataset, test_indices),
        batch_size=16,
        shuffle=False,
        num_workers=4,
        pin_memory=(device.type == "cuda"),
    )

    model = VCFDeepfakeDetector(pretrained=False).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    all_labels = []
    all_logits = {"raw": [], "c23": [], "c40": []}

    with torch.no_grad():
        for raw, c23, c40, labels, _group_ids in test_loader:
            raw, c23, c40 = raw.to(device), c23.to(device), c40.to(device)

            # autocast is applied inside the model's forward
            lr, _ = model(raw)
            l23, _ = model(c23)
            l40, _ = model(c40)

            all_labels.extend(labels.numpy().astype(int).tolist())
            all_logits["raw"].extend(lr.float().cpu().numpy().reshape(-1).tolist())
            all_logits["c23"].extend(l23.float().cpu().numpy().reshape(-1).tolist())
            all_logits["c40"].extend(l40.float().cpu().numpy().reshape(-1).tolist())

    y_true = np.array(all_labels)
    logits = {k: np.array(v) for k, v in all_logits.items()}

    # Overall prediction: average logits across the three compression versions
    avg_logits = (logits["raw"] + logits["c23"] + logits["c40"]) / 3.0
    probs = 1.0 / (1.0 + np.exp(-avg_logits))
    y_pred = (probs >= 0.5).astype(int)

    print("\n===== OVERALL TEST PERFORMANCE =====")
    print(f"Checkpoint epoch: {checkpoint['epoch']}")
    print(f"Test samples: {len(y_true)}")
    print(f"Accuracy:          {accuracy_score(y_true, y_pred):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision:         {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:            {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1 score:          {f1_score(y_true, y_pred, zero_division=0):.4f}")

    print("\nConfusion matrix (rows=true, columns=predicted), 0=real, 1=fake:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred, labels=[0, 1],
        target_names=["Real", "Fake"], zero_division=0,
    ))

    # Accuracy of each compression version on its own
    per_version = {}
    print("Per-version accuracy / balanced accuracy:")
    for name in ("raw", "c23", "c40"):
        pred_v = (logits[name] >= 0).astype(int)
        acc = accuracy_score(y_true, pred_v)
        bal = balanced_accuracy_score(y_true, pred_v)
        per_version[name] = {"accuracy": float(acc), "balanced_accuracy": float(bal)}
        print(f"  {name}: {acc:.4f} / {bal:.4f}")

    # Save results
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    summary = {
        "checkpoint_epoch": int(checkpoint["epoch"]),
        "test_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
        "per_version": per_version,
    }
    with open(RESULTS_DIR / "test_metrics.json", "w") as f:
        json.dump(summary, f, indent=2)

    with open(RESULTS_DIR / "test_predictions.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["relative_path", "label", "prob_fake", "predicted"])
        for idx, lab, pr, pd_ in zip(test_indices, y_true, probs, y_pred):
            w.writerow([dataset.records[idx]["relative_path"], int(lab), f"{pr:.6f}", int(pd_)])

    print(f"\nSaved: {RESULTS_DIR / 'test_metrics.json'}")
    print(f"Saved: {RESULTS_DIR / 'test_predictions.csv'}")


evaluate_test()

Device: cuda
Checkpoint: /kaggle/working/checkpoints/best_model_linear_fixed_split.pt
Cache: /kaggle/input/datasets/thirteenthdev/preprocessed-videos-cache
Video file counts:
  raw: 1600
  c23: 1600
  c40: 1600
  common relative paths: 1600
  raw missing c23: 0
  raw missing c40: 0
  c23 files without raw: 0
  c40 files without raw: 0
Matched records retained: 1600 | real=640 | fake=960

===== OVERALL TEST PERFORMANCE =====
Checkpoint epoch: 1
Test samples: 400
Accuracy:          0.6075
Balanced accuracy: 0.5427
Precision:         0.6246
Recall:            0.8667
F1 score:          0.7260

Confusion matrix (rows=true, columns=predicted), 0=real, 1=fake:
[[ 35 125]
 [ 32 208]]

Classification report:
              precision    recall  f1-score   support

        Real       0.52      0.22      0.31       160
        Fake       0.62      0.87      0.73       240

    accuracy                           0.61       400
   macro avg       0.57      0.54      0.52       400
weighted avg       